# Pipeline Huấn Luyện & Đánh Giá Validation LightGBM (ĐẦY ĐỦ 35 Đặc Trưng) - Loss MAE
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

Notebook này thực hiện trọn vẹn quy trình (**Tune Optuna -> Train Final -> Evaluate Validation**) cho **Loss MAE** trên **CẢ 2 HORIZON (h1: t+1 và h4: t+4)**:
- **Cấu hình đặc trưng:** `Full 35 features (có lag_1)`
- **h1 (t+1):** Dự báo 15 phút tiếp theo
- **h4 (t+4):** Dự báo 1 giờ tiếp theo (4 bước 15 phút)

> ### Lưu ý về Hỗ Trợ 2 Horizon (t+1 và t+4) & Thư Mục Đầu Ra
>
> **ĐƯỜNG DẪN XUẤT KẾT QUẢ:** Kết quả được lưu tại `../../data/model/v3/06_train/mae/`
> **NGUYÊN TẮC BAN ĐÊM VÀ NIÊM PHONG TẬP TEST:**
> 1. Mô hình vẫn được **huấn luyện trên toàn bộ ngày + đêm** để học thời điểm chuyển giao bình minh/hoàng hôn.
> 2. **METRIC CHÍNH THỨC BÁO CÁO** chỉ tính ở phạm vi BAN NGÀY (`is_daylight == True` và `energy_source == "measured"`).
> 3. Notebook này **KHÔNG ĐỤNG TỚI TẬP TEST**. Chấm test duy nhất 1 lần tại Notebook 07.

## Bước 2. Import thư viện và khai báo tham số

In [ ]:
import gc
import json
import os
import pickle
import platform
import statistics
import time
import warnings

import lightgbm as lgb
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
import numpy as np
import optuna
import pandas as pd
import pyarrow.parquet as pq
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Tham số chung ──
VERSION = 'v3'
SITE_COL = 'site_id'
TIMESTAMP_COL = 'timestamp'
TARGET_COL = 'energy_generated_kwh'
LOSS_NAME = 'mae'
LGB_OBJECTIVE = 'regression_l1'

EXCLUDE_FEATURES = []
FEATURE_SET_NAME = ""

HORIZONS = [1, 4]       # h1 (t+1: 15m tới) và h4 (t+4: 1h tới)
FOLDS = [1, 2, 3, 4, 5]
N_TRIALS = 20
SEED = 42
EARLY_STOPPING_ROUNDS = 100

USE_GPU = True          # Đổi thành False nếu muốn ép chạy CPU
GPU_PLATFORM_ID = 0
GPU_DEVICE_ID = 0
VERBOSE_FOLD = True     # In tiến độ từng fold trong mỗi trial Optuna

# ── Thư mục đầu vào / đầu ra ──
SELECTED_DIR = '../../data/model/v3/05_selected'
BASE_OUTPUT_DIR = '../../data/model/v3/06_train'
FOLDER_NAME = f"{LOSS_NAME}_{FEATURE_SET_NAME}" if FEATURE_SET_NAME else LOSS_NAME
OUTPUT_DIR = f'{BASE_OUTPUT_DIR}/{FOLDER_NAME}'

os.makedirs(OUTPUT_DIR, exist_ok=True)
for h in HORIZONS:
    os.makedirs(f'{OUTPUT_DIR}/h{h}', exist_ok=True)

print("Đã import thư viện và khai báo tham số.")
print(f"- Loss function      : {LOSS_NAME.upper()} (objective: {LGB_OBJECTIVE})")
print(f"- Feature set name   : {FEATURE_SET_NAME if FEATURE_SET_NAME else 'full'}")
print(f"- Excluded features  : {EXCLUDE_FEATURES}")
print(f"- Horizons hỗ trợ    : {HORIZONS} (h1 = t+1 / 15m; h4 = t+4 / 1h)")
print(f"- Cấu hình GPU       : USE_GPU={USE_GPU} (platform_id={GPU_PLATFORM_ID}, device_id={GPU_DEVICE_ID})")
print(f"- Log chi tiết fold  : VERBOSE_FOLD={VERBOSE_FOLD}")
print(f"- Số trials Optuna   : {N_TRIALS} mỗi horizon")
print(f"- Đọc dữ liệu từ      : {SELECTED_DIR}")
print(f"- Ghi kết quả ra      : {OUTPUT_DIR}")

## Bước 2.1. Thiết lập GPU (OpenCL) cho LightGBM

In [ ]:
LA_LINUX = (os.name == "posix" and platform.system() == "Linux")

if LA_LINUX:
    OCL_CANDIDATES = [
        "/run/opengl-driver/etc/OpenCL/vendors",
        "/etc/OpenCL/vendors",
    ]
    if "OCL_ICD_VENDORS" not in os.environ:
        for _p in OCL_CANDIDATES:
            if os.path.isdir(_p) and any(f.endswith(".icd") for f in os.listdir(_p)):
                os.environ["OCL_ICD_VENDORS"] = _p
                print("Đã tự đặt OCL_ICD_VENDORS = " + str(_p))
                break


def kiem_tra_gpu():
    try:
        X = np.random.rand(100, 4)
        y = np.random.rand(100)
        lgb.train(
            {"objective": "regression", "device": "gpu", "gpu_platform_id": GPU_PLATFORM_ID, "gpu_device_id": GPU_DEVICE_ID, "verbose": -1},
            lgb.Dataset(X, y),
            num_boost_round=1
        )
        return True, ""
    except Exception as e:
        return False, str(e)[:200]

GPU_SAN_SANG = False
if USE_GPU:
    GPU_SAN_SANG, _err = kiem_tra_gpu()
    if GPU_SAN_SANG:
        print("GPU OpenCL sẵn sàng. LightGBM sẽ chạy trên GPU.")
    else:
        print("[CẢNH BÁO] Không dùng được GPU, tự động chuyển sang CPU.")
        print("Lý do: " + str(_err))

print("Chế độ tính toán chính thức: " + ("GPU" if GPU_SAN_SANG else "CPU"))

## Bước 3. Đọc danh sách đặc trưng đã chọn

In [ ]:
json_path = f'{SELECTED_DIR}/selected_features.json'
with open(json_path, 'r', encoding='utf-8') as f:
    _sel_raw = json.load(f)

selected_features = _sel_raw['selected_features'] if isinstance(_sel_raw, dict) else _sel_raw

_before = len(selected_features)
selected_features = [c for c in selected_features if c not in EXCLUDE_FEATURES]
if EXCLUDE_FEATURES:
    print("Bỏ " + str(_before - len(selected_features)) + " đặc trưng: " + str(EXCLUDE_FEATURES))
print("Số đặc trưng sử dụng: " + str(len(selected_features)))

NEEDED_COLS = selected_features + [
    TARGET_COL, SITE_COL, TIMESTAMP_COL,
    "energy_source", "exclude_from_training",
    "outlier_group", "has_complete_history_features", "is_daylight"
]


def read_selected(path):
    have = set(pq.ParquetFile(path).schema_arrow.names)
    cols = [c for c in NEEDED_COLS if c in have]
    return pd.read_parquet(path, columns=cols)


def add_horizon_target(df, horizon_steps):
    out = df.copy()
    target_col_name = f'target_h{horizon_steps}'
    if horizon_steps == 1:
        out[target_col_name] = out[TARGET_COL]
    else:
        shift_steps = -(horizon_steps - 1)
        out[target_col_name] = out.groupby(SITE_COL)[TARGET_COL].shift(shift_steps)
    return out, target_col_name


print(f"Đã định nghĩa hàm read_selected & add_horizon_target cho {len(selected_features)} đặc trưng.")

## Bước 4. Hàm lọc dòng hợp lệ

In [ ]:
def filter_valid_rows(df, name="", target_col=TARGET_COL):
    n_before = len(df)
    out = df.copy()

    if 'exclude_from_training' in out.columns:
        out = out[out['exclude_from_training'] == False]

    if 'has_complete_history_features' in out.columns:
        out = out[out['has_complete_history_features'] == True]

    if target_col in out.columns:
        out = out[out[target_col].notna()]

    n_after = len(out)
    n_removed = n_before - n_after
    pct = (n_after / n_before) * 100 if n_before > 0 else 0
    print(f"Lọc dữ liệu {name} (target={target_col}): ban đầu {n_before:,} dòng -> còn {n_after:,} dòng (loại {n_removed:,} dòng, {pct:.2f}%)")
    return out


print("Đã định nghĩa hàm filter_valid_rows linh hoạt theo target_col.")

## Bước 5. Chạy Trọn Vẹn Pipeline Huấn Luyện Cho Cả 2 Horizon (h1 & h4)

In [ ]:
all_val_metrics = {}

for h in HORIZONS:
    h_label = f"h{h}"
    h_desc = "t+1 (15m tới)" if h == 1 else "t+4 (1h tới)"
    h_dir = f"{OUTPUT_DIR}/{h_label}"
    os.makedirs(h_dir, exist_ok=True)

    print("")
    print("=" * 80)
    print(f"=== BẮT ĐẦU PIPELINE CHO HORIZON {h_label.upper()} ({h_desc}) | LOSS = {LOSS_NAME.upper()} | FEATURE_SET = {FEATURE_SET_NAME if FEATURE_SET_NAME else 'full'} ===")
    print("=" * 80)

    cached_folds = []
    for fold in FOLDS:
        tr_path = f'{SELECTED_DIR}/time_series_folds/fold_{fold}_train_selected.parquet'
        va_path = f'{SELECTED_DIR}/time_series_folds/fold_{fold}_val_selected.parquet'

        if not os.path.exists(tr_path) or not os.path.exists(va_path):
            continue

        tr_raw = read_selected(tr_path)
        va_raw = read_selected(va_path)

        tr_raw, t_col = add_horizon_target(tr_raw, h)
        va_raw, t_col = add_horizon_target(va_raw, h)

        tr_df = filter_valid_rows(tr_raw, f"Fold {fold} Train", target_col=t_col)
        va_df = filter_valid_rows(va_raw, f"Fold {fold} Val", target_col=t_col)
        del tr_raw, va_raw
        gc.collect()

        feat_cols = [c for c in selected_features if c in tr_df.columns]
        num_cols = [c for c in feat_cols if pd.api.types.is_numeric_dtype(tr_df[c])]
        feat_cols = num_cols
        cat_cols = [c for c in feat_cols if c.endswith('_enc')]

        medians = tr_df[feat_cols].median(numeric_only=True).fillna(0.0)

        x_tr = tr_df[feat_cols].fillna(medians).astype(np.float32)
        y_tr = tr_df[t_col].astype(np.float32)
        x_va = va_df[feat_cols].fillna(medians).astype(np.float32)
        y_va = va_df[t_col].astype(np.float32)

        cached_folds.append({
            'fold': fold,
            'x_train': x_tr,
            'y_train': y_tr,
            'x_val': x_va,
            'y_val': y_va,
            'cat_cols': cat_cols,
        })
        del tr_df, va_df
        gc.collect()

    print(f"Đã nạp {len(cached_folds)} fold cho Horizon {h_label.upper()} vào bộ nhớ ({len(feat_cols)} đặc trưng).")

    def objective(trial):
        obj_choice = LGB_OBJECTIVE
        reg_type = trial.suggest_categorical("reg_type", ["l1", "l2", "elasticnet"])
        if reg_type == "l1":
            reg_alpha = trial.suggest_float("reg_alpha", 0.0, 10.0)
            reg_lambda = 0.0
        elif reg_type == "l2":
            reg_alpha = 0.0
            reg_lambda = trial.suggest_float("reg_lambda", 0.0, 10.0)
        else:
            reg_alpha = trial.suggest_float("reg_alpha", 0.0, 10.0)
            reg_lambda = trial.suggest_float("reg_lambda", 0.0, 10.0)

        n_est_max = trial.suggest_int("n_estimators", 200, 800)
        learning_rate = trial.suggest_float("learning_rate", 0.01, 0.15, log=True)
        num_leaves = trial.suggest_int("num_leaves", 31, 127)
        min_child_samples = trial.suggest_int("min_child_samples", 20, 200)
        subsample = trial.suggest_float("subsample", 0.7, 1.0)
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.7, 1.0)

        params = {
            'objective': obj_choice,
            'n_estimators': n_est_max,
            'learning_rate': learning_rate,
            'num_leaves': num_leaves,
            'min_child_samples': min_child_samples,
            'subsample': subsample,
            'colsample_bytree': colsample_bytree,
            'reg_alpha': reg_alpha,
            'reg_lambda': reg_lambda,
            'random_state': SEED,
            'n_jobs': -1,
            'verbosity': -1,
        }
        if obj_choice == "huber":
            params['alpha'] = trial.suggest_float("alpha", 0.5, 10.0, log=True)

        if GPU_SAN_SANG:
            params['device'] = 'gpu'
            params['gpu_platform_id'] = GPU_PLATFORM_ID
            params['gpu_device_id'] = GPU_DEVICE_ID

        abs_err_sum, abs_y_sum = 0.0, 0.0
        fold_best_iterations = []

        for f_idx, f_payload in enumerate(cached_folds, 1):
            t_fold_start = time.time()
            model = LGBMRegressor(**params)
            cat_cols = f_payload['cat_cols']

            try:
                model.fit(
                    f_payload['x_train'], f_payload['y_train'],
                    eval_set=[(f_payload['x_val'], f_payload['y_val'])],
                    eval_metric='l1',
                    callbacks=[early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False), log_evaluation(period=0)],
                    categorical_feature=cat_cols if cat_cols else 'auto',
                )
            except Exception as e:
                if params.get('device') == 'gpu':
                    cpu_params = params.copy()
                    cpu_params['device'] = 'cpu'
                    cpu_params.pop('gpu_platform_id', None)
                    cpu_params.pop('gpu_device_id', None)
                    model = LGBMRegressor(**cpu_params)
                    model.fit(
                        f_payload['x_train'], f_payload['y_train'],
                        eval_set=[(f_payload['x_val'], f_payload['y_val'])],
                        eval_metric='l1',
                        callbacks=[early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False), log_evaluation(period=0)],
                        categorical_feature=cat_cols if cat_cols else 'auto',
                    )
                else:
                    raise e

            best_iter = int(getattr(model, "best_iteration_", None) or params["n_estimators"])
            fold_best_iterations.append(best_iter)

            pred = model.predict(f_payload['x_val'], num_iteration=best_iter)
            y_val = f_payload['y_val'].to_numpy(dtype=float)

            f_err = float(np.nansum(np.abs(y_val - pred)))
            f_y = float(np.nansum(np.abs(y_val)))
            abs_err_sum += f_err
            abs_y_sum += f_y

            fold_wape = (f_err / f_y * 100.0) if f_y > 0 else float("nan")
            t_fold_elapsed = time.time() - t_fold_start
            if VERBOSE_FOLD:
                print(f"      [{h_label}] fold {f_idx}/5 | WAPE {fold_wape:.3f}% | best_iter {best_iter} | {t_fold_elapsed:.1f}s")

        if fold_best_iterations:
            trial.set_user_attr("best_iteration_median", int(statistics.median(fold_best_iterations)))

        pooled_wape = (abs_err_sum / abs_y_sum * 100.0) if abs_y_sum > 0 else float("inf")
        trial.report(pooled_wape, step=trial.number)
        if trial.should_prune():
            raise optuna.TrialPruned()

        return pooled_wape

    print(f"--- BẮT ĐẦU TUNE OPTUNA FOR {h_label.upper()} ({h_desc}) ---")
    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED), pruner=MedianPruner())
    _t0_study = time.time()

    def log_trial(study, trial):
        elapsed = time.time() - _t0_study
        done = trial.number + 1
        tb_min = elapsed / 60.0
        eta_min = (elapsed / done) * (N_TRIALS - done) / 60.0 if done > 0 else 0.0
        try:
            best_val = study.best_value
        except ValueError:
            best_val = float("inf")
        is_new_best = (trial.value is not None and abs(trial.value - best_val) < 1e-12)
        mark_best = " [MỚI TỐT NHẤT]" if is_new_best else ""
        val_str = f"{trial.value:.4f}%" if (trial.state != optuna.trial.TrialState.PRUNED and trial.value is not None) else "(bị prune)"
        p = trial.params
        print(f"[{h_label.upper()} Trial {done:>2}/{N_TRIALS}] WAPE {val_str:>12} | Best {best_val:.4f}% | reg={p.get('reg_type')} lr={p.get('learning_rate', 0):.4f} | {tb_min:.1f}m, còn ~{eta_min:.1f}m{mark_best}")

    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False, callbacks=[log_trial])

    best_iter_median = study.best_trial.user_attrs.get("best_iteration_median")
    final_n_estimators = int(best_iter_median) if best_iter_median else int(study.best_params.get("n_estimators", 500))

    trials_df = study.trials_dataframe()
    trials_df.to_csv(f'{h_dir}/optuna_trials.csv', index=False)

    best_params_export = {
        'horizon_steps': h,
        'loss_name': LOSS_NAME,
        'feature_set_name': FEATURE_SET_NAME,
        'excluded_features': EXCLUDE_FEATURES,
        'num_features': len(feat_cols),
        'objective': LGB_OBJECTIVE,
        'best_pooled_wape': float(study.best_value),
        'final_n_estimators': final_n_estimators,
        'best_params': study.best_params,
    }
    with open(f'{h_dir}/best_params.json', 'w', encoding='utf-8') as f:
        json.dump(best_params_export, f, ensure_ascii=False, indent=2)

    del cached_folds
    gc.collect()

    print(f"--- TRAIN FINAL MODEL FOR {h_label.upper()} ({h_desc}) ---")
    dev_path = f'{SELECTED_DIR}/{VERSION}_development_selected.parquet'
    dev_raw = read_selected(dev_path)
    dev_raw, t_col = add_horizon_target(dev_raw, h)
    dev_df = filter_valid_rows(dev_raw, f"Development {h_label.upper()}", target_col=t_col)
    del dev_raw
    gc.collect()

    feat_cols = [c for c in selected_features if c in dev_df.columns]
    num_cols = [c for c in feat_cols if pd.api.types.is_numeric_dtype(dev_df[c])]
    feat_cols = num_cols
    cat_cols = [c for c in feat_cols if c.endswith('_enc')]

    feature_medians = dev_df[feat_cols].median(numeric_only=True).fillna(0.0)
    X_dev = dev_df[feat_cols].fillna(feature_medians).astype(np.float32)
    y_dev = dev_df[t_col].astype(np.float32)

    best_p = study.best_params.copy()
    reg_type = best_p.pop('reg_type', None)
    best_p.pop('n_estimators', None)
    reg_alpha = best_p.pop('reg_alpha', 0.0)
    reg_lambda = best_p.pop('reg_lambda', 0.0)

    final_params = {
        'objective': LGB_OBJECTIVE,
        'n_estimators': final_n_estimators,
        'reg_alpha': reg_alpha,
        'reg_lambda': reg_lambda,
        'random_state': SEED,
        'n_jobs': -1,
        'verbosity': -1,
        **best_p
    }
    if GPU_SAN_SANG:
        final_params['device'] = 'gpu'
        final_params['gpu_platform_id'] = GPU_PLATFORM_ID
        final_params['gpu_device_id'] = GPU_DEVICE_ID

    t_train_start = time.time()
    final_model = LGBMRegressor(**final_params)
    try:
        final_model.fit(X_dev, y_dev, categorical_feature=cat_cols if cat_cols else 'auto')
    except Exception as e:
        if final_params.get('device') == 'gpu':
            cpu_params = final_params.copy()
            cpu_params['device'] = 'cpu'
            cpu_params.pop('gpu_platform_id', None)
            cpu_params.pop('gpu_device_id', None)
            final_model = LGBMRegressor(**cpu_params)
            final_model.fit(X_dev, y_dev, categorical_feature=cat_cols if cat_cols else 'auto')
        else:
            raise e

    with open(f'{h_dir}/model.pkl', 'wb') as f:
        pickle.dump(final_model, f)

    model_config_payload = {
        'horizon_steps': h,
        'loss_name': LOSS_NAME,
        'feature_set_name': FEATURE_SET_NAME,
        'excluded_features': EXCLUDE_FEATURES,
        'lgb_objective': LGB_OBJECTIVE,
        'final_n_estimators': final_n_estimators,
        'train_rows': len(dev_df),
        'features': feat_cols,
        'feature_medians': feature_medians.to_dict(),
        'model_params': final_params,
    }
    with open(f'{h_dir}/model_config.json', 'w', encoding='utf-8') as f:
        json.dump(model_config_payload, f, ensure_ascii=False, indent=2)

    del dev_df, X_dev, y_dev
    gc.collect()

    print(f"--- EVALUATE VALIDATION FOR {h_label.upper()} ({h_desc}) ---")
    val_path = f'{SELECTED_DIR}/{VERSION}_val_selected.parquet'
    val_raw = read_selected(val_path)
    val_raw, t_col = add_horizon_target(val_raw, h)
    val_df = filter_valid_rows(val_raw, f"Validation {h_label.upper()}", target_col=t_col)
    del val_raw
    gc.collect()

    X_val = val_df[feat_cols].fillna(feature_medians).astype(float)
    y_true = val_df[t_col].values
    y_pred = final_model.predict(X_val)

    def compute_wape_func(yt, yp):
        abs_y = np.nansum(np.abs(yt))
        return (np.nansum(np.abs(yt - yp)) / abs_y * 100.0) if abs_y > 0 else np.nan

    def compute_metrics_func(yt, yp):
        return {
            'wape': compute_wape_func(yt, yp),
            'rmse': root_mean_squared_error(yt, yp),
            'mae': mean_absolute_error(yt, yp),
            'r2': r2_score(yt, yp),
        }

    m_all = compute_metrics_func(y_true, y_pred)
    mask_meas = (val_df['energy_source'] == 'measured').values if 'energy_source' in val_df.columns else np.ones(len(val_df), dtype=bool)
    m_meas = compute_metrics_func(y_true[mask_meas], y_pred[mask_meas]) if mask_meas.sum() > 0 else {'wape': np.nan, 'rmse': np.nan, 'mae': np.nan, 'r2': np.nan}

    if 'is_daylight' in val_df.columns:
        mask_day = (val_df['is_daylight'] == True).values | (val_df['is_daylight'] == 1).values
        mask_meas_day = mask_meas & mask_day
    else:
        mask_meas_day = np.zeros(len(val_df), dtype=bool)

    m_meas_day = compute_metrics_func(y_true[mask_meas_day], y_pred[mask_meas_day]) if mask_meas_day.sum() > 0 else {'wape': np.nan, 'rmse': np.nan, 'mae': np.nan, 'r2': np.nan}

    metrics_val_payload = {
        'horizon_steps': h,
        'loss_name': LOSS_NAME,
        'feature_set_name': FEATURE_SET_NAME,
        'excluded_features': EXCLUDE_FEATURES,
        'lgb_objective': LGB_OBJECTIVE,
        'pooled_wape_cv': float(study.best_value),
        'val_total_rows': len(val_df),
        'val_measured_rows': int(mask_meas.sum()),
        'val_measured_daylight_rows': int(mask_meas_day.sum()),
        'all': m_all,
        'measured': m_meas,
        'measured_daylight': m_meas_day,
        'scope_c_measured_daylight_official': m_meas_day,
    }
    with open(f'{h_dir}/metrics_val.json', 'w', encoding='utf-8') as f:
        json.dump(metrics_val_payload, f, ensure_ascii=False, indent=2)

    all_val_metrics[h_label] = metrics_val_payload
    print(f"[{h_label.upper()} HOÀN TẤT] Validation Measured&Daylight WAPE: {m_meas_day['wape']:.2f}% | RMSE: {m_meas_day['rmse']:.4f}")

    del val_df, X_val, y_true, y_pred
    gc.collect()

with open(f'{OUTPUT_DIR}/metrics_val.json', 'w', encoding='utf-8') as f:
    json.dump(all_val_metrics, f, ensure_ascii=False, indent=2)

print("")
print("=" * 80)
print(f"=== TỔNG KẾT HOÀN TẤT CẢ 2 HORIZON CHO LOSS {LOSS_NAME.upper()} (FEATURE_SET={{FEATURE_SET_NAME if FEATURE_SET_NAME else 'full'}}) ===")
print("=" * 80)

### Bảo vệ Niêm phong Tập Test:
1. Notebook này chỉ huấn luyện và đánh giá trên tập Validation cho CẢ 2 HORIZON.
2. Tập Test chỉ được đưa vào chấm 1 lần duy nhất tại `07_final_test.ipynb`.